# Electric Bus Charging Problem with Aggregator Support

In [1]:
import pyomo.environ as pyo
from pyomo.opt import SolverFactory
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

EXEC_PATH = '/Applications/CPLEX_Studio221/cplex/bin/x86-64_osx/cplex'

In [2]:
data = pd.read_excel('../Instances/input.xlsx', None)

## HPR

In [3]:
def solveHRP(data,y_buy,y_sell,y_cap,d_l,u_l,count,scenario):

    model = pyo.ConcreteModel()  # create model

    # data
    t = len(data['Prices']['Spot Market'])
    PI = data['Prices']['Spot Market'].values.flatten() 
    PI_cap = data['Prices']['Capacity price'].values.flatten() 
    l = len(data['Power price']['Power'])

    X_up = data['Average prices']['Max price'].values.flatten() 
    X_low = data['Average prices']['Min price'].values.flatten() 
    X_avg = 0.12
    Mi_up = data['Average prices']['Max cap'].values.flatten() 
    Mi_low = data['Average prices']['Min cap'].values.flatten() 
    Mi_avg = 0.01
    
    p = len(data['Periods']['Period'])
    Q_begin = data['Periods']['Begin'].tolist()
    Q_end = data['Periods']['End'].tolist()
    Q_len = data['Periods']['Len'].tolist()

    i = len(data['Trip time']['Time begin (min)'])
    k = len(data['Buses']['Bus (kWh)'])
    n = len(data['Chargers']['Charger (kWh/min)'])
    T_start = data['Trip time']['Time begin (min)'].tolist()
    T_start = [int(x) for x in T_start]
    T_end = data['Trip time']['Time finish (min)'].tolist()
    T_end = [int(x) for x in T_end]
    alpha = data['Chargers']['Charger (kWh/min)'].tolist()
    beta = data['Chargers']['Charger (kWh/min)'].tolist()
    gama = data['Energy consumption']['Uncertain energy (kWh/km*min)'].tolist()
    ch_eff = 0.90
    dch_eff = 1
    E_0 = 0.2
    E_min = 0.2
    E_max = 1
    E_end = 0.2
    C_bat = data['Buses']['Bus (kWh)'].tolist()

    U_pow = data['Power price']['Power'].tolist()
    U_price = data['Power price']['Price'].tolist()
    U_max = data['Chargers']['Max Power (kW)'].tolist()
    U_max = U_max[0]
    U_cap = 950

    d_on = 1
    d_off = 1
    d_cap = 1

    R = 130  # 130
    Ah = 905452  # 905452
    V = 512
    T = 96
    delta_t = 4
    delta = 0.8
    M = 10000

    # sets
    model.P = pyo.RangeSet(p)  # set of periods
    model.I = pyo.RangeSet(i)  # set of trips
    model.T = pyo.RangeSet(t)  # set of timesteps
    model.K = pyo.RangeSet(k)  # set of buses
    model.N = pyo.RangeSet(n)  # set of chargers
    model.L = pyo.RangeSet(l)  # set of chargers

    # parameters
    model.PI = pyo.Param(model.T, initialize=lambda model,t: PI[t-1])  # spot market electricity prices
    model.PI_cap = pyo.Param(model.T, initialize=lambda model,t: PI_cap[t-1])  # capacity market electricity prices
    model.Q_begin = pyo.Param(model.P, initialize=lambda model, p: Q_begin[p-1]) # Period Q_p starting time
    model.Q_end = pyo.Param(model.P, initialize=lambda model, p: Q_end[p-1]) # Period Q_p ending time
    model.Q_len = pyo.Param(model.P, initialize=lambda model, p: Q_len[p-1]) # Period Q_p length
    model.X_low = pyo.Param(model.P, initialize=lambda model, p: X_low[p-1]) # Mininum value price x can achieve
    model.X_up = pyo.Param(model.P, initialize=lambda model, p: X_up[p-1]) # Maxinum value price x can achieve
    model.X_avg = pyo.Param(initialize=(X_avg)) # Average price x can achieve
    model.Mi_low = pyo.Param(model.P, initialize=lambda model, p: Mi_low[p-1]) # Mininum value price mi can achieve
    model.Mi_up = pyo.Param(model.P, initialize=lambda model, p: Mi_up[p-1]) # Maxinum value price mi can achieve
    model.Mi_avg = pyo.Param(initialize=(Mi_avg)) # Average price mi can achieve
    model.T_start = pyo.Param(model.I, initialize=lambda model, i: T_start[i-1])  # start time of trip i
    model.T_end = pyo.Param(model.I, initialize=lambda model, i: T_end[i-1])  # end time of trip i
    model.alpha = pyo.Param(model.N, initialize=lambda model, n: alpha[n-1]) # charging power of charger n
    model.beta = pyo.Param(model.N, initialize=lambda model,n: beta[n-1]) # discharging power of charger n
    model.ch_eff = pyo.Param(initialize=ch_eff) # charging efficiency of charger n
    model.dch_eff = pyo.Param(initialize=dch_eff) # discharging efficiency of charger n
    model.gama = pyo.Param(model.I, initialize=lambda model, i: gama[i-1], mutable=True)  # energy consumption
    model.E_0 = pyo.Param(initialize=E_0)  # initial energy level of bus k
    model.E_min = pyo.Param(initialize=E_min) # minimum energy level allowed for bus k
    model.E_max = pyo.Param(initialize=E_max) # maximum energy level allowed for bus k
    model.E_end = pyo.Param(initialize=E_end) # minimum energy after an operation day for bus k
    model.C_bat = pyo.Param(model.K, initialize=lambda model, k: C_bat[k-1]) # total capacity of the bus k battery
    model.U_pow = pyo.Param(model.L, initialize=lambda model, l: U_pow[l-1])  # power level l
    model.U_price = pyo.Param(model.L, initialize=lambda model, l: U_price[l-1]) # purchasing price for power level l
    model.U_max = pyo.Param(initialize=U_max) # contracted power
    model.U_cap = pyo.Param(initialize=U_cap) # mininum power capacity for bid
    model.delta = pyo.Param(initialize=delta) # difference between sell/buy
    model.R = pyo.Param(initialize=R)  # battery replacement costs of the bus k
    model.Ah = pyo.Param(initialize=Ah)  # energy consumed until EOL of bus k
    model.V = pyo.Param(initialize=V)  # operational voltage of charger n

    #parameters from LL
    model.y_buy = pyo.Param(model.T, initialize=y_buy)
    if scenario == 2 or 4:
        model.y_sell = pyo.Param(model.T, initialize=y_sell)
        model.d_l = pyo.Param(model.K,model.T,initialize=d_l)
    if scenario == 3 or 4:
        model.y_cap = pyo.Param(model.T, initialize=y_cap)
    model.u_l = pyo.Param(model.L, initialize=u_l)

    # non-negative variables
    model.e = pyo.Var(model.K, model.T, within=pyo.NonNegativeReals) # energy level of bus k at time t
    model.w_buy = pyo.Var(model.T, within=pyo.NonNegativeReals) # electricity purchased from the grid at time t
    
    if scenario == 2 or 4:
        model.w_sell = pyo.Var(model.T, within=pyo.NonNegativeReals) # electricity purchased from the grid at time t
        model.d = pyo.Var(model.K, model.T, within=pyo.NonNegativeReals) # total degradation cost of the bus k battery at time t
    if scenario == 3 or 4:
        model.w_cap = pyo.Var(model.T, within=pyo.NonNegativeReals) # electricity sold to the grid at time t
        model.a = pyo.Var(model.T, domain=pyo.Binary) # binary variable indicating if the mininum power bid constraint habe been meet at time t

    # binary variables
    model.b = pyo.Var(model.K, model.I, model.T, within=pyo.Binary) # binary variable indicating if bus k is serving trip i at time t
    model.x = pyo.Var(model.K, model.N, model.T, domain=pyo.Binary) # binary variable indicating if bus k is charging
    if scenario == 2 or 4:
        model.y = pyo.Var(model.K, model.N, model.T, domain=pyo.Binary) # binary variable indicating if bus k is discharging
    
    model.c = pyo.Var(model.K, model.T, domain=pyo.Binary) # binary variable indicating if bus k is parked to charge at time t
    model.u = pyo.Var(model.L, domain=pyo.Binary) # binary variable indicating the peak power level l
    model.pho_plus = pyo.Var(model.P, domain=pyo.NonNegativeIntegers) # electricity purchasing price at period p
    model.pho_minus = pyo.Var(model.P, domain=pyo.NonNegativeIntegers) # electricity selling price at period p
    model.mi = pyo.Var(model.P, domain=pyo.NonNegativeIntegers) # capacity selling price at period p

    if scenario == 3 or 4:
        model.z = pyo.Var(model.K, model.N, model.T, domain=pyo.Binary) # binary variable indicating if bus k is offering capacity to the grid at time t
        model.z_up = pyo.Var(model.K, model.N, model.T, domain=pyo.Binary) # binary variable indicating if bus k is occupying a charger to provide capacity to the grid
        model.z_down = pyo.Var(model.K, model.N, model.T, domain=pyo.Binary) # binary variable indicating if bus k is occupying a charger to provide capacity to the grid


    # constraints
    model.constraints = pyo.ConstraintList()
    ### UL ###
    # constraint 5
    for p in model.P:
        model.constraints.add(model.pho_plus[p] >= model.X_low[p])

    for p in model.P:
        model.constraints.add(model.pho_plus[p] <= model.X_up[p])

    # constraint 6
    if scenario == 2 or 4:
        for p in model.P:
            model.constraints.add(
                model.pho_minus[p] == model.pho_plus[p] * model.delta)

    # constraint 8
    if scenario == 3 or 4:
        for p in model.P:
            model.constraints.add(model.mi[p] >= model.Mi_low[p])

        for p in model.P:
            model.constraints.add(model.mi[p] <= model.Mi_up[p])

        # constraint 7
        model.constraints.add(((1/T) * sum(model.Q_len[p] * model.mi[p] for p in model.P)) <= model.Mi_avg)

    # constraint 7
    model.constraints.add(((1/T) * sum(model.Q_len[p] * model.mi[p] for p in model.P)) <= model.X_avg)

    ### LL ###
    #constraint 14
    for k in model.K:
        for t in model.T:
            model.constraints.add(sum(model.b[k, i, t]
                                for i in model.I) + model.c[k, t] <= 1)

    #constraint 15
    for i in model.I:
        for t in range(model.T_start[i], model.T_end[i]):
            model.constraints.add(sum(model.b[k, i, t] for k in model.K) == 1)

    #constraint 16
    for i in model.I:
        for k in model.K:
            for t in range(model.T_start[i], model.T_end[i]-1):
                model.constraints.add(model.b[k, i, t+1] >= model.b[k, i, t])

    #############################################################################################################################################################################
    if scenario == 1:
        #constraint 17
        for n in model.N:
            for t in model.T:
                model.constraints.add(sum(model.x[k, n, t] for k in model.K) <= 1)
    
    if scenario == 2:
        #constraint 17
        for n in model.N:
            for t in model.T:
                model.constraints.add(sum(model.x[k, n, t] for k in model.K) + sum(model.y[k, n, t] for k in model.K) <= 1)

    if scenario == 3:
        #constraint 17
        for n in model.N:
            for t in model.T:
                model.constraints.add(sum(model.x[k, n, t] for k in model.K) + sum(model.z[k, n, t] for k in model.K) <= 1)

    if scenario == 4:
        #constraint 17
        for n in model.N:
            for t in model.T:
                model.constraints.add(sum(model.x[k, n, t] for k in model.K) + sum(model.y[k, n, t] for k in model.K) + sum(model.z[k, n, t] for k in model.K) <= 1)
    #############################################################################################################################################################################
    if scenario == 1:
        #constraint 18
        for k in model.K:
            for t in model.T:
                model.constraints.add(sum(model.x[k, n, t] for n in model.N) <= model.c[k, t])

    if scenario == 2:
        #constraint 18
        for k in model.K:
            for t in model.T:
                model.constraints.add(sum(model.x[k, n, t] for n in model.N) + sum(model.y[k, n, t] for n in model.N) <= model.c[k, t])

    if scenario == 3:
        #constraint 18
        for k in model.K:
            for t in model.T:
                model.constraints.add(sum(model.x[k, n, t] for n in model.N) + sum(model.z[k, n, t] for n in model.N) <= model.c[k, t])

    if scenario == 4:
        #constraint 18
        for k in model.K:
            for t in model.T:
                model.constraints.add(sum(model.x[k, n, t] for n in model.N) + sum(model.y[k, n, t] for n in model.N) + sum(model.z[k, n, t] for n in model.N) <= model.c[k, t])

    #############################################################################################################################################################################
    if scenario == 1:
        #constraint 19
        for k in model.K:
            for t in range(2, T+1):
                model.constraints.add(model.e[k, t] == model.e[k, t-1] + sum(model.ch_eff*model.alpha[n]*model.x[k, n, t] for n in model.N) - sum(model.gama[i]*model.b[k, i, t] for i in model.I))

    if scenario == 2:
        #constraint 19
        for k in model.K:
            for t in range(2, T+1):
                model.constraints.add(model.e[k, t] == model.e[k, t-1] + sum(model.ch_eff*model.alpha[n]*model.x[k, n, t] for n in model.N) - sum(model.gama[i]*model.b[k, i, t] for i in model.I) - sum(model.dch_eff*model.beta[n]*model.y[k, n, t] for n in model.N))

    if scenario == 3:
        #constraint 19
        for k in model.K:
            for t in range(2, T+1):
                model.constraints.add(model.e[k, t] == model.e[k, t-1] + sum(model.ch_eff*model.alpha[n]*model.x[k, n, t] for n in model.N) - sum(model.gama[i]*model.b[k, i, t] for i in model.I))

    if scenario == 4:
        #constraint 19
        for k in model.K:
            for t in range(2, T+1):
                model.constraints.add(model.e[k, t] == model.e[k, t-1] + sum(model.ch_eff*model.alpha[n]*model.x[k, n, t] for n in model.N) - sum(model.gama[i]*model.b[k, i, t] for i in model.I) - sum(model.dch_eff*model.beta[n]*model.y[k, n, t] for n in model.N))

    #############################################################################################################################################################################

    #constraint 20
    for t in model.T:
        model.constraints.add(sum(model.ch_eff*model.alpha[n]*model.x[k, n, t] for n in model.N for k in model.K) == model.w_buy[t])
    
    if scenario == 2 or 4:
        #constraint 21.1
        for t in model.T:
            model.constraints.add(sum(
                model.dch_eff*model.beta[n]*model.y[k, n, t] for n in model.N for k in model.K) == model.w_sell[t])
        #constraint 21.2
        model.constraints.add(sum(model.dch_eff*model.beta[n]*model.y[k, n, 1] for n in model.N for k in model.K) == 0)
    
    '''
    #constraint 22
    for k in model.K:
        for n in model.N:
            for t in range(2, T-d_on):
                model.constraints.add(1 - model.x[k, n, t] + model.x[k, n, t-1] + ((1/d_on)*sum(model.x[k, n, j] for j in range(t, t+d_on))) >= 1)

    #constraint 23
    for k in model.K:
        for n in model.N:
            for t in range(T-d_on+1, T):
                model.constraints.add(1 - model.x[k, n, t] + model.x[k, n, t-1] + ((1/(T-t+1))*sum(model.x[k, n, j] for j in range(t, T))) >= 1)

    #constraint 24
    for k in model.K:
        for n in model.N:
            for t in range(2, T-d_off):
                model.constraints.add(1 - model.x[k, n, t] + model.x[k, n, t-1] + ((1/d_off)*sum(model.x[k, n, j] for j in range(t, t+d_off))) <= 2)

    #constraint 25
    for k in model.K:
        for n in model.N:
            for t in range(T-d_off+1, T):
                model.constraints.add(1 - model.x[k, n, t] + model.x[k, n, t-1] + ((1/(T-t+1))*sum(model.x[k, n, j] for j in range(t, T))) <= 2)
    '''
    
    #constraint 26
    model.constraints.add(sum(model.u[l] for l in model.L) == 1)

    #constraint 27
    for t in model.T:
        model.constraints.add(sum(model.alpha[n]*model.x[k, n, t] for k in model.K for n in model.N) <= sum(model.U_pow[l]*model.u[l] for l in model.L))

    #constraint 28
    for t in model.T:
        model.constraints.add(sum(model.alpha[n]*model.x[k, n, t] for k in model.K for n in model.N) <= model.U_max)

    #constraint 29
    for k in model.K:
        for t in model.T:
            model.constraints.add(model.e[k, t] >= model.C_bat[k] * model.E_min)

    #constrait 30
    for k in model.K:
        for t in model.T:
            model.constraints.add(E_max * model.C_bat[k] >= model.e[k, t] + sum(model.ch_eff*model.alpha[n]*model.x[k, n, t] for n in model.N))

    #constraint 31
    for k in model.K:
        model.constraints.add(model.e[k, 1] == model.E_0*model.C_bat[k])

    #constraint 32
    for k in model.K:
        model.constraints.add(model.e[k, T-1] + sum(model.ch_eff*model.alpha[n] * model.x[k, n, T] for n in model.N) >= model.E_end*model.C_bat[k])

    if scenario == 2 or 4:    
        #constraint 33
        for k in model.K:
            for t in model.T:
                model.constraints.add(model.d[k, t] == ((model.R*model.C_bat[1]*1000)/(4*model.Ah*model.V)) * (sum(model.dch_eff*model.beta[n]* model.y[k, n, t] for n in model.N)))

    if scenario == 3 or 4:
        #constraint 34
        for n in model.N:
            for t in model.T:
                model.constraints.add(sum(model.z_up[k, n, t] for k in model.K)+sum(model.z_down[k, n, t] for k in model.K)==sum(model.z[k, n, t] for k in model.K))
        
        #constraint 35
        for t in model.T:
            model.constraints.add(sum(model.ch_eff*model.alpha[n] * model.z_down[k, n, t] for k in model.K for n in model.N) * delta_t + sum(model.dch_eff*model.beta[n] * model.z_up[k, n, t] for k in model.K for n in model.N) * delta_t == model.w_cap[t])

        #constrait 36
        for k in model.K:
            for t in model.T:
                model.constraints.add(E_max * model.C_bat[k] >= model.e[k, t] + sum(model.ch_eff*model.alpha[n]*model.z_down[k, n, t] for n in model.N))
        
        #constrait 37
        for k in model.K:
            for t in model.T:
                model.constraints.add(E_min * model.C_bat[k]<= model.e[k, t] - sum(model.dch_eff*model.beta[n] * model.z_up[k, n, t] for k in model.K for n in model.N))
        
        #constraint 38
        for t in model.T:
            model.constraints.add(model.w_cap[t] >= model.U_cap - M*(1-model.a[t]))

        #constraint 39
        for t in model.T:
            model.constraints.add(model.w_cap[t] <=  M*model.a[t])
        
        #constraint 40
        model.constraints.add(sum(model.z_up[k, n, t]for k in model.K for n in model.N for t in model.T)==sum(model.z_down[k, n, t]for k in model.K for n in model.N for t in model.T))
    
        '''
        #constraint 41
        for k in model.K:
            for n in model.N:
                for t in range(2, T-d_cap):
                    model.constraints.add(1 - model.z[k, n, t] + model.z[k, n, t-1] + ((1/d_cap)*sum(model.z[k, n, j] for j in range(t, t+d_cap))) >= 1)

        #constraint 42
        for k in model.K:
            for n in model.N:
                for t in range(T-d_cap+1, T):
                    model.constraints.add(1 - model.z[k, n, t] + model.z[k, n, t-1] + ((1/(T-t+1))*sum(model.z[k, n, j] for j in range(t, T))) >= 1)
        '''
    
    if count > 1:

        if scenario == 1:

            #constraint HPR
            model.constraints.add(
                sum(model.pho_plus[p]*model.w_buy[t] for p in model.P for t in range(model.Q_begin[p], model.Q_end[p])) +
                sum(model.U_price[l]*model.u[l] for l in model.L)       
                <=
                sum(model.pho_plus[p]*model.y_buy[t] for p in model.P for t in range(model.Q_begin[p], model.Q_end[p])) +
                sum(model.U_price[l]*model.u_l[l] for l in model.L)
            )

        if scenario == 2:

            #constraint HPR
            model.constraints.add(
                sum(model.pho_plus[p]*model.w_buy[t] for p in model.P for t in range(model.Q_begin[p], model.Q_end[p])) -
                sum(model.pho_minus[p]*model.w_sell[t] for p in model.P for t in range(model.Q_begin[p], model.Q_end[p])) +
                sum(model.d[k,t] for k in model.K for t in model.T) +
                sum(model.U_price[l]*model.u[l] for l in model.L)       
                <=
                sum(model.pho_plus[p]*model.y_buy[t] for p in model.P for t in range(model.Q_begin[p], model.Q_end[p])) -
                sum(model.pho_minus[p]*model.y_sell[t] for p in model.P for t in range(model.Q_begin[p], model.Q_end[p])) +
                sum(model.d_l[k, t] for k in model.K for t in model.T) +
                sum(model.U_price[l]*model.u_l[l] for l in model.L)
            )

        if scenario == 3:

            #constraint HPR
            model.constraints.add(
                sum(model.pho_plus[p]*model.w_buy[t] for p in model.P for t in range(model.Q_begin[p], model.Q_end[p])) -
                sum(model.mi[p]*model.w_cap[t] for p in model.P for t in range(model.Q_begin[p], model.Q_end[p])) +
                sum(model.U_price[l]*model.u[l] for l in model.L)       
                <=
                sum(model.pho_plus[p]*model.y_buy[t] for p in model.P for t in range(model.Q_begin[p], model.Q_end[p])) -
                sum(model.mi[p]*model.y_cap[t] for p in model.P for t in range(model.Q_begin[p], model.Q_end[p])) +
                sum(model.U_price[l]*model.u_l[l] for l in model.L)
            )
            
        if scenario == 4:

            #constraint HPR
            model.constraints.add(
                sum(model.pho_plus[p]*model.w_buy[t] for p in model.P for t in range(model.Q_begin[p], model.Q_end[p])) -
                sum(model.pho_minus[p]*model.w_sell[t] for p in model.P for t in range(model.Q_begin[p], model.Q_end[p])) -
                sum(model.mi[p]*model.w_cap[t] for p in model.P for t in range(model.Q_begin[p], model.Q_end[p])) +
                sum(model.d[k,t] for k in model.K for t in model.T) +
                sum(model.U_price[l]*model.u[l] for l in model.L)       
                <=
                sum(model.pho_plus[p]*model.y_buy[t] for p in model.P for t in range(model.Q_begin[p], model.Q_end[p])) -
                sum(model.pho_minus[p]*model.y_sell[t] for p in model.P for t in range(model.Q_begin[p], model.Q_end[p])) -
                sum(model.mi[p]*model.y_cap[t] for p in model.P for t in range(model.Q_begin[p], model.Q_end[p])) +
                sum(model.d_l[k, t] for k in model.K for t in model.T) +
                sum(model.U_price[l]*model.u_l[l] for l in model.L)
            )
    
    # Objective function
    if scenario == 1:
        def rule_obj(mod):
            return sum(mod.pho_plus[p] * mod.w_buy[t] for p in mod.P for t in range(mod.Q_begin[p], mod.Q_end[p])) - sum(mod.PI[t] * mod.w_buy[t] for t in mod.T)   
    if scenario == 2:
        def rule_obj(mod):    
            return sum(mod.pho_plus[p] * mod.w_buy[t] for p in mod.P for t in range(mod.Q_begin[p], mod.Q_end[p])) - sum(mod.pho_minus[p] * mod.w_sell[t] for p in mod.P for t in range(mod.Q_begin[p], mod.Q_end[p]))  + sum(mod.PI[t] * mod.w_sell[t] for t in mod.T) - sum(mod.PI[t] * mod.w_buy[t] for t in mod.T)        
    if scenario == 3:
        def rule_obj(mod):
            return sum(mod.pho_plus[p] * mod.w_buy[t] for p in mod.P for t in range(mod.Q_begin[p], mod.Q_end[p])) + sum(mod.PI_cap[t] * mod.w_cap[t] for p in mod.P for t in range(mod.Q_begin[p], mod.Q_end[p])) - sum( mod.mi[p] * mod.w_cap[t] for p in mod.P for t in range(mod.Q_begin[p], mod.Q_end[p])) - sum(mod.PI[t] * mod.w_buy[t] for t in mod.T)
    if scenario == 4:
        def rule_obj(mod):
            return sum(mod.pho_plus[p] * mod.w_buy[t] for p in mod.P for t in range(mod.Q_begin[p], mod.Q_end[p])) - sum(mod.pho_minus[p] * mod.w_sell[t] for p in mod.P for t in range(mod.Q_begin[p], mod.Q_end[p])) + sum(mod.PI_cap[t] * mod.w_cap[t] for p in mod.P for t in range(mod.Q_begin[p], mod.Q_end[p])) - sum( mod.mi[p] * mod.w_cap[t] for p in mod.P for t in range(mod.Q_begin[p], mod.Q_end[p]))  + sum(mod.PI[t] * mod.w_sell[t] for t in mod.T) - sum(mod.PI[t] * mod.w_buy[t] for t in mod.T)
    model.obj = pyo.Objective(rule=rule_obj, sense=pyo.maximize)
    
    opt = pyo.SolverFactory('ipopt')
    solver_manager = pyo.SolverManagerFactory('neos')
    solver_manager.solve(model, opt=opt)

    return model

## LL

In [4]:

def solveLL(data,pho_plus,pho_minus,mi,scenario):

    model = pyo.ConcreteModel()

    i = len(data['Trip time']['Time begin (min)'])
    t = 96
    k = len(data['Buses']['Bus (kWh)'])
    n = len(data['Chargers']['Charger (kWh/min)'])
    l = len(data['Power price']['Power'])
    p = len(data['Periods']['Period'])

    T_start = data['Trip time']['Time begin (min)'].tolist()
    T_start = [int(x) for x in T_start]
    T_end = data['Trip time']['Time finish (min)'].tolist()
    T_end = [int(x) for x in T_end]

    Q_begin = data['Periods']['Begin'].tolist()  # Period Q_p starting time
    Q_begin = [int(x) for x in Q_begin]
    Q_end = data['Periods']['End'].tolist()  # Period Q_p ending time
    Q_end = [int(x) for x in Q_end]

    alpha = data['Chargers']['Charger (kWh/min)'].tolist()
    beta = data['Chargers']['Charger (kWh/min)'].tolist()
    ch_eff = 0.90
    dch_eff = 1
    gama = data['Energy consumption']['Uncertain energy (kWh/km*min)'].tolist()

    E_0 = 0.2
    E_min = 0.2
    E_max = 1
    E_end = 0.2
    C_bat = data['Buses']['Bus (kWh)'].tolist()

    d_off = 1
    d_on = 1
    d_cap = 1

    U_pow = data['Power price']['Power'].tolist()
    U_price = data['Power price']['Price'].tolist()
    U_max = data['Chargers']['Max Power (kW)'].tolist()
    U_max = U_max[0]
    U_cap = 950

    R = 130
    Ah = 905452
    V = 512
    T = 96
    delta_t = 4
    M = 10000

    ## Sets
    model.I = pyo.RangeSet(i)  # set of trips
    model.T = pyo.RangeSet(t)  # set of timesteps
    model.K = pyo.RangeSet(k)  # set of buses
    model.N = pyo.RangeSet(n)  # set of chargers
    model.L = pyo.RangeSet(l)  # set of peak power levels
    model.P = pyo.RangeSet(p)  # number of price periods
    
    ## Parameters
    model.T_start = pyo.Param(model.I, initialize=lambda model, i: T_start[i-1])  # start time of trip i
    model.T_end = pyo.Param(model.I, initialize=lambda model, i: T_end[i-1])  # end time of trip i
    model.alpha = pyo.Param(model.N, initialize=lambda model, n: alpha[n-1])  # charging power of charger n
    model.beta = pyo.Param(model.N, initialize=lambda model, n: beta[n-1])  # discharging power of charger n
    model.ch_eff = pyo.Param(initialize=ch_eff)  # charging efficiency of charger n
    model.dch_eff = pyo.Param(initialize=dch_eff) # discharging efficiency of charger n
    model.gama = pyo.Param(model.I, initialize=lambda model, i: gama[i-1])  # energy consumption
    model.E_0 = pyo.Param(initialize=E_0)  # initial energy level of bus k
    model.E_min = pyo.Param(initialize=E_min) # minimum energy level allowed for bus k
    model.E_max = pyo.Param(initialize=E_max) # maximum energy level allowed for bus k
    model.E_end = pyo.Param(initialize=E_end) # minimum energy after an operation day for bus k
    model.C_bat = pyo.Param(model.K, initialize=lambda model, k: C_bat[k-1]) # total capacity of the bus k battery
    model.U_pow = pyo.Param(model.L, initialize=lambda model, l: U_pow[l-1]) # power level l
    model.U_price = pyo.Param(model.L, initialize=lambda model, l: U_price[l-1]) # purchasing price for power level l
    model.U_max = pyo.Param(initialize=U_max)  # contracted power
    model.U_cap = pyo.Param(initialize=U_cap) # mininum bid power
    model.R = pyo.Param(initialize=R)  # battery replacement costs of the bus k
    model.Ah = pyo.Param(initialize=Ah)  # energy consumed until EOL of bus k
    model.V = pyo.Param(initialize=V)  # operational voltage of charger n
    model.Q_begin = pyo.Param(model.P, initialize=lambda model, p: Q_begin[p-1]) # Period Q_p starting time
    model.Q_end = pyo.Param(model.P, initialize=lambda model, p: Q_end[p-1]) # Period Q_p ending time
    model.pho_plus = pyo.Param(model.P, initialize=pho_plus) # electricity purchasing price in time t
    model.pho_minus = pyo.Param(model.P, initialize=pho_minus) # electricity selling price in time t
    model.mi = pyo.Param(model.P, initialize=mi) # electricity purchasing price in time t

    # Decision Variables
    # binary variables
    model.b = pyo.Var(model.K, model.I, model.T, within=pyo.Binary) # binary variable indicating if bus k is serving trip i at time t
    model.x = pyo.Var(model.K, model.N, model.T, domain=pyo.Binary) # binary variable indicating if bus k is occupying a charger n at time t to charge
    if scenario == 2 or 4:
        model.y = pyo.Var(model.K, model.N, model.T, domain=pyo.Binary) # binary variable indicating if bus k is occupying a charger n at time t to discharge
    model.u = pyo.Var(model.L, domain=pyo.Binary) # binary variable indicating the peak power level l
    model.c = pyo.Var(model.K, model.T, domain=pyo.Binary) # binary variable indicating if bus k is charging/discharging at time t

    if scenario == 3 or 4:
        model.z = pyo.Var(model.K, model.N, model.T, domain=pyo.Binary) # binary variable indicating if bus k is occupying a charger to provide capacity to the grid
        model.z_up = pyo.Var(model.K, model.N, model.T, domain=pyo.Binary) # binary variable indicating if bus k is occupying a charger to provide capacity to the grid
        model.z_down = pyo.Var(model.K, model.N, model.T, domain=pyo.Binary) # binary variable indicating if bus k is occupying a charger to provide capacity to the grid
        model.a = pyo.Var(model.T, domain=pyo.Binary) # binary variable indicating if the mininum power bid constraint habe been meet at time t

    #non-negative variables
    model.e = pyo.Var(model.K, model.T, within=pyo.NonNegativeReals) # energy level of bus k at time t
    model.w_buy = pyo.Var(model.T, within=pyo.NonNegativeReals) # electricity purchased from the grid at time t
    if scenario == 2 or 4:
        model.w_sell = pyo.Var(model.T, within=pyo.NonNegativeReals) # electricity sold to the grid at time t
        model.d = pyo.Var(model.K, model.T, within=pyo.NonNegativeReals) # total degradation cost of the bus k battery at time t
    if scenario == 3 or 4:
        model.w_cap = pyo.Var(model.T, within=pyo.NonNegativeReals) # electricity sold to the grid at time t

    ## Constraints
    model.constraints = pyo.ConstraintList()  # Create a set of constraints

### LL ###
    #constraint 14
    for k in model.K:
        for t in model.T:
            model.constraints.add(sum(model.b[k, i, t]
                                for i in model.I) + model.c[k, t] <= 1)

    #constraint 15
    for i in model.I:
        for t in range(model.T_start[i], model.T_end[i]):
            model.constraints.add(sum(model.b[k, i, t] for k in model.K) == 1)

    #constraint 16
    for i in model.I:
        for k in model.K:
            for t in range(model.T_start[i], model.T_end[i]-1):
                model.constraints.add(model.b[k, i, t+1] >= model.b[k, i, t])

    #############################################################################################################################################################################
    if scenario == 1:
        #constraint 17
        for n in model.N:
            for t in model.T:
                model.constraints.add(sum(model.x[k, n, t] for k in model.K) <= 1)
    
    if scenario == 2:
        #constraint 17
        for n in model.N:
            for t in model.T:
                model.constraints.add(sum(model.x[k, n, t] for k in model.K) + sum(model.y[k, n, t] for k in model.K) <= 1)

    if scenario == 3:
        #constraint 17
        for n in model.N:
            for t in model.T:
                model.constraints.add(sum(model.x[k, n, t] for k in model.K) + sum(model.z[k, n, t] for k in model.K) <= 1)

    if scenario == 4:
        #constraint 17
        for n in model.N:
            for t in model.T:
                model.constraints.add(sum(model.x[k, n, t] for k in model.K) + sum(model.y[k, n, t] for k in model.K) + sum(model.z[k, n, t] for k in model.K) <= 1)
    #############################################################################################################################################################################
    if scenario == 1:
        #constraint 18
        for k in model.K:
            for t in model.T:
                model.constraints.add(sum(model.x[k, n, t] for n in model.N) <= model.c[k, t])

    if scenario == 2:
        #constraint 18
        for k in model.K:
            for t in model.T:
                model.constraints.add(sum(model.x[k, n, t] for n in model.N) + sum(model.y[k, n, t] for n in model.N) <= model.c[k, t])

    if scenario == 3:
        #constraint 18
        for k in model.K:
            for t in model.T:
                model.constraints.add(sum(model.x[k, n, t] for n in model.N) + sum(model.z[k, n, t] for n in model.N) <= model.c[k, t])

    if scenario == 4:
        #constraint 18
        for k in model.K:
            for t in model.T:
                model.constraints.add(sum(model.x[k, n, t] for n in model.N) + sum(model.y[k, n, t] for n in model.N) + sum(model.z[k, n, t] for n in model.N) <= model.c[k, t])

    #############################################################################################################################################################################
    if scenario == 1:
        #constraint 19
        for k in model.K:
            for t in range(2, T+1):
                model.constraints.add(model.e[k, t] == model.e[k, t-1] + sum(model.ch_eff*model.alpha[n]*model.x[k, n, t] for n in model.N) - sum(model.gama[i]*model.b[k, i, t] for i in model.I))

    if scenario == 2:
        #constraint 19
        for k in model.K:
            for t in range(2, T+1):
                model.constraints.add(model.e[k, t] == model.e[k, t-1] + sum(model.ch_eff*model.alpha[n]*model.x[k, n, t] for n in model.N) - sum(model.gama[i]*model.b[k, i, t] for i in model.I) - sum(model.dch_eff*model.beta[n]*model.y[k, n, t] for n in model.N))

    if scenario == 3:
        #constraint 19
        for k in model.K:
            for t in range(2, T+1):
                model.constraints.add(model.e[k, t] == model.e[k, t-1] + sum(model.ch_eff*model.alpha[n]*model.x[k, n, t] for n in model.N) - sum(model.gama[i]*model.b[k, i, t] for i in model.I))

    if scenario == 4:
        #constraint 19
        for k in model.K:
            for t in range(2, T+1):
                model.constraints.add(model.e[k, t] == model.e[k, t-1] + sum(model.ch_eff*model.alpha[n]*model.x[k, n, t] for n in model.N) - sum(model.gama[i]*model.b[k, i, t] for i in model.I) - sum(model.dch_eff*model.beta[n]*model.y[k, n, t] for n in model.N))

    #############################################################################################################################################################################

    #constraint 20
    for t in model.T:
        model.constraints.add(sum(model.ch_eff*model.alpha[n]*model.x[k, n, t] for n in model.N for k in model.K) == model.w_buy[t])
    
    if scenario == 2 or 4:
        #constraint 21.1
        for t in model.T:
            model.constraints.add(sum(
                model.dch_eff*model.beta[n]*model.y[k, n, t] for n in model.N for k in model.K) == model.w_sell[t])
        #constraint 21.2
        model.constraints.add(sum(model.dch_eff*model.beta[n]*model.y[k, n, 1] for n in model.N for k in model.K) == 0)
    
    '''
    #constraint 22
    for k in model.K:
        for n in model.N:
            for t in range(2, T-d_on):
                model.constraints.add(1 - model.x[k, n, t] + model.x[k, n, t-1] + ((1/d_on)*sum(model.x[k, n, j] for j in range(t, t+d_on))) >= 1)

    #constraint 23
    for k in model.K:
        for n in model.N:
            for t in range(T-d_on+1, T):
                model.constraints.add(1 - model.x[k, n, t] + model.x[k, n, t-1] + ((1/(T-t+1))*sum(model.x[k, n, j] for j in range(t, T))) >= 1)

    #constraint 24
    for k in model.K:
        for n in model.N:
            for t in range(2, T-d_off):
                model.constraints.add(1 - model.x[k, n, t] + model.x[k, n, t-1] + ((1/d_off)*sum(model.x[k, n, j] for j in range(t, t+d_off))) <= 2)

    #constraint 25
    for k in model.K:
        for n in model.N:
            for t in range(T-d_off+1, T):
                model.constraints.add(1 - model.x[k, n, t] + model.x[k, n, t-1] + ((1/(T-t+1))*sum(model.x[k, n, j] for j in range(t, T))) <= 2)
    '''
    
    #constraint 26
    model.constraints.add(sum(model.u[l] for l in model.L) == 1)

    #constraint 27
    for t in model.T:
        model.constraints.add(sum(model.alpha[n]*model.x[k, n, t] for k in model.K for n in model.N) <= sum(model.U_pow[l]*model.u[l] for l in model.L))

    #constraint 28
    for t in model.T:
        model.constraints.add(sum(model.alpha[n]*model.x[k, n, t] for k in model.K for n in model.N) <= model.U_max)

    #constraint 29
    for k in model.K:
        for t in model.T:
            model.constraints.add(model.e[k, t] >= model.C_bat[k] * model.E_min)

    #constrait 30
    for k in model.K:
        for t in model.T:
            model.constraints.add(E_max * model.C_bat[k] >= model.e[k, t] + sum(model.ch_eff*model.alpha[n]*model.x[k, n, t] for n in model.N))

    #constraint 31
    for k in model.K:
        model.constraints.add(model.e[k, 1] == model.E_0*model.C_bat[k])

    #constraint 32
    for k in model.K:
        model.constraints.add(model.e[k, T-1] + sum(model.ch_eff*model.alpha[n] * model.x[k, n, T] for n in model.N) >= model.E_end*model.C_bat[k])

    if scenario == 2 or 4:    
        #constraint 33
        for k in model.K:
            for t in model.T:
                model.constraints.add(model.d[k, t] == ((model.R*model.C_bat[1]*1000)/(4*model.Ah*model.V)) * (sum(model.dch_eff*model.beta[n]* model.y[k, n, t] for n in model.N)))

    if scenario == 3 or 4:
        #constraint 34
        for n in model.N:
            for t in model.T:
                model.constraints.add(sum(model.z_up[k, n, t] for k in model.K)+sum(model.z_down[k, n, t] for k in model.K)==sum(model.z[k, n, t] for k in model.K))
        
        #constraint 35
        for t in model.T:
            model.constraints.add(sum(model.ch_eff*model.alpha[n] * model.z_down[k, n, t] for k in model.K for n in model.N) * delta_t + sum(model.dch_eff*model.beta[n] * model.z_up[k, n, t] for k in model.K for n in model.N) * delta_t == model.w_cap[t])

        #constrait 36
        for k in model.K:
            for t in model.T:
                model.constraints.add(E_max * model.C_bat[k] >= model.e[k, t] + sum(model.ch_eff*model.alpha[n]*model.z_down[k, n, t] for n in model.N))
        
        #constrait 37
        for k in model.K:
            for t in model.T:
                model.constraints.add(E_min * model.C_bat[k]<= model.e[k, t] - sum(model.dch_eff*model.beta[n] * model.z_up[k, n, t] for k in model.K for n in model.N))
        
        #constraint 38
        for t in model.T:
            model.constraints.add(model.w_cap[t] >= model.U_cap - M*(1-model.a[t]))

        #constraint 39
        for t in model.T:
            model.constraints.add(model.w_cap[t] <=  M*model.a[t])
        
        #constraint 40
        model.constraints.add(sum(model.z_up[k, n, t]for k in model.K for n in model.N for t in model.T)==sum(model.z_down[k, n, t]for k in model.K for n in model.N for t in model.T))
    
        '''
        #constraint 41
        for k in model.K:
            for n in model.N:
                for t in range(2, T-d_cap):
                    model.constraints.add(1 - model.z[k, n, t] + model.z[k, n, t-1] + ((1/d_cap)*sum(model.z[k, n, j] for j in range(t, t+d_cap))) >= 1)

        #constraint 42
        for k in model.K:
            for n in model.N:
                for t in range(T-d_cap+1, T):
                    model.constraints.add(1 - model.z[k, n, t] + model.z[k, n, t-1] + ((1/(T-t+1))*sum(model.z[k, n, j] for j in range(t, T))) >= 1)
        '''

    # Objective Function
    if scenario == 1:
        def rule_obj(mod):
            return sum(mod.pho_plus[p]*mod.w_buy[t] for p in mod.P for t in range(mod.Q_begin[p], mod.Q_end[p])) + sum(mod.U_price[l]*mod.u[l] for l in mod.L)
    if scenario == 2:
        def rule_obj(mod):
            return sum(mod.pho_plus[p]*mod.w_buy[t] for p in mod.P for t in range(mod.Q_begin[p], mod.Q_end[p])) - sum(mod.pho_minus[p]*mod.w_sell[t] for p in mod.P for t in range(mod.Q_begin[p], mod.Q_end[p])) + sum(mod.d[k, t] for k in mod.K for t in mod.T) + sum(mod.U_price[l]*mod.u[l] for l in mod.L)
    if scenario == 3:
        def rule_obj(mod):
            return sum(mod.pho_plus[p]*mod.w_buy[t] for p in mod.P for t in range(mod.Q_begin[p], mod.Q_end[p])) + sum(mod.U_price[l]*mod.u[l] for l in mod.L) - sum(mod.mi[p]*mod.w_cap[t] for p in mod.P for t in range(mod.Q_begin[p], mod.Q_end[p]))
    if scenario == 4:
        def rule_obj(mod):
            return sum(mod.pho_plus[p]*mod.w_buy[t] for p in mod.P for t in range(mod.Q_begin[p], mod.Q_end[p])) - sum(mod.pho_minus[p]*mod.w_sell[t] for p in mod.P for t in range(mod.Q_begin[p], mod.Q_end[p])) + sum(mod.d[k, t] for k in mod.K for t in mod.T) + sum(mod.U_price[l]*mod.u[l] for l in mod.L) - sum(mod.mi[p]*mod.w_cap[t] for p in mod.P for t in range(mod.Q_begin[p], mod.Q_end[p]))
    model.obj = pyo.Objective(rule=rule_obj, sense=pyo.minimize)

    opt = pyo.SolverFactory('gurobi')
    opt.options['timelimit'] = 1800
    opt.options['mipgap'] = 0.001
    opt.solve(model,tee=False)
    
    return model

# Vizualization

In [5]:
def visualization(model_LL, model_HRP,pho_plus,pho_minus,mi,y_buy_l,y_sell_l,y_cap_l,scenario):
    
    # Values PTO
    buy = sum(pho_plus[p]*model_LL.w_buy[t] for p in model_LL.P for t in range(model_LL.Q_begin[p], model_LL.Q_end[p]))
    if scenario == 2 or 4:
        sell = sum(pho_minus[p]*model_LL.w_sell[t] for p in model_LL.P for t in range(model_LL.Q_begin[p], model_LL.Q_end[p]))
        degra = sum(model_LL.d[k, t] for k in model_LL.K for t in model_LL.T)
    else:
        sell = 0
        degra = 0
    power = sum(model_LL.U_price[l]*model_LL.u[l] for l in model_LL.L)
    if scenario == 3 or 4:
        cap = sum(mi[p]*model_LL.w_cap[t] for p in model_LL.P for t in range(model_LL.Q_begin[p], model_LL.Q_end[p]))
    else:
        cap = 0
    costs = buy - sell + degra + power - cap
    print('\n''Values from the PTO side:\n'
        'Total costs:',pyo.value(costs),'\n'
        'Energy bought from the Aggregator:',pyo.value(buy),'\n'
        'Energy sold to the Aggregator:',pyo.value(sell),'\n'
        'Battery degradation costs:',pyo.value(degra),'\n'
        'Power bought:',pyo.value(power),'\n'
        'FCR offered to the Aggregator:',pyo.value(cap),'\n')

    #Values Aggregator
    sell_to_PTO = sum(pho_plus[p] * y_buy_l[t] for p in model_HRP.P for t in range(model_HRP.Q_begin[p], model_HRP.Q_end[p]))

    if scenario == 2 or 4:
        buy_from_PTO = sum(pho_minus[p] * y_sell_l[t] for p in model_HRP.P for t in range(model_HRP.Q_begin[p], model_HRP.Q_end[p]))
        sell_to_GRID = sum(model_HRP.PI[t] * y_sell_l[t] for t in model_HRP.T)
    else:
        buy_from_PTO = 0
        sell_to_GRID = 0

    if scenario == 3 or 4:
        cap_from_PTO = sum(mi[p] * y_cap_l[t] for p in model_HRP.P for t in range(model_HRP.Q_begin[p], model_HRP.Q_end[p]))
        cap_to_GRID = sum(model_HRP.PI_cap[t] * y_cap_l[t] for p in model_HRP.P for t in range(model_HRP.Q_begin[p], model_HRP.Q_end[p]))
    else:
        cap_from_PTO = 0
        cap_to_GRID = 0
    buy_from_GRID = sum(model_HRP.PI[t] * y_buy_l[t] for t in model_HRP.T)
    revenues = sell_to_PTO - buy_from_PTO - cap_from_PTO + cap_to_GRID - buy_from_GRID + sell_to_GRID

    print('Values from the Aggregator side:\n'
        'Total revenues:',pyo.value(revenues),'\n'
        'Energy sold to the PTO:',pyo.value(sell_to_PTO),'\n'
        'Energy bought from the PTO:',pyo.value(buy_from_PTO),'\n'
        'FCR bought from the PTO:',pyo.value(cap_from_PTO),'\n'
        'FCR offered to the grid:',pyo.value(cap_to_GRID),'\n'
        'Energy bought in the Wholesale market:',pyo.value(buy_from_GRID),'\n'
        'Energy sold to the grid:',pyo.value(sell_to_GRID))
    
    # Initialize empty lists
    pho_plus_data = []
    pho_minus_data = []
    mi_data = []
    spot_data = []
    min_data = []
    max_data = []
    mi_min_data = []
    mi_max_data = []

    # Getting prices information
    for p in model_HRP.P:
        for t in range(model_HRP.Q_begin[p], model_HRP.Q_end[p]):
            pho_plus_data.append(pyo.value(pho_plus[p]))
            pho_minus_data.append(pyo.value(pho_minus[p]))
            mi_data.append(pyo.value(mi[p]))
            spot_data.append(pyo.value(model_HRP.PI[t]))
            min_data.append(pyo.value(model_HRP.X_low[p]))
            max_data.append(pyo.value(model_HRP.X_up[p]))  
            mi_min_data.append(pyo.value(model_HRP.Mi_low[p]))
            mi_max_data.append(pyo.value(model_HRP.Mi_up[p]))

    df = pd.DataFrame({'sell': pho_plus_data, 'buy': pho_minus_data,
                    'cap': mi_data, 'spot': spot_data, 'min': min_data, 'max': max_data, 'min_cap': mi_min_data, 'max_cap': mi_max_data})

    def extract_model_data(model):
        w_buy = [model.w_buy[t]() * 4 for t in model.T]
        w_sell = [model.w_sell[t]() * 4 for t in model.T]
        w_cap = [model.w_cap[t]() for t in model.T]
        e_values = np.array([[model.e[k, t].value for t in model.T] for k in model.K])
        u_values = [model.u[l].value for l in model.L]
        x_values = np.array([[[model.x[k, n, t].value for t in model.T] for n in model.N] for k in model.K])
        y_values = np.array([[[model.y[k, n, t].value for t in model.T] for n in model.N] for k in model.K])
        z_values = np.array([[[model.z[k, n, t].value for t in model.T] for n in model.N] for k in model.K])
        c_values = np.array([[model.c[k, t].value for t in model.T] for k in model.K])
        d_values = np.array([[model.d[k, t].value for t in model.T] for k in model.K])
        b_values = np.array([[[model.b[k, i, t].value for t in model.T] for i in model.I] for k in model.K])

        return w_buy, w_sell, w_cap, e_values, u_values, x_values, y_values, z_values, c_values, d_values, b_values

    # Extract model data
    w_buy, w_sell, w_cap, e_values, u_values, x_values, y_values, z_values, c_values, d_values, b_values = extract_model_data(model_LL)

    # Construct the output file name based on the scenario variable
    if scenario == 1:
        output_file_name = 'output_data_scenario_1.xlsx'
    elif scenario == 2:
        output_file_name = 'output_data_scenario_2.xlsx'
    elif scenario == 3:
        output_file_name = 'output_data_scenario_3.xlsx'
    elif scenario == 4:
        output_file_name = 'output_data_scenario_4.xlsx'
    else:
        raise ValueError(f"Unsupported scenario: {scenario}")


    # Create DataFrame for the first set of data
    data1 = {'sell': pho_plus_data, 'buy': pho_minus_data,
            'cap': mi_data, 'spot': spot_data, 'min': min_data, 'max': max_data, 'min_cap': mi_min_data, 'max_cap': mi_max_data}
    df1 = pd.DataFrame(data1)

    # Save the first DataFrame to an Excel file
    with pd.ExcelWriter(output_file_name) as writer:
        df1.to_excel(writer, sheet_name='Tariffs', index=False)

    # Create DataFrames for the second set of data
    data2 = {'w_buy': w_buy, 'w_sell': w_sell, 'w_cap': w_cap}
    df2 = pd.DataFrame(data2)

    # Create DataFrame for e_values
    e_values_t = e_values.T  # transposing the data
    e_df = pd.DataFrame(e_values_t, columns=model_LL.K, index=model_LL.T)

    # Save the second set of DataFrames to the same Excel file, each in a separate sheet
    with pd.ExcelWriter(output_file_name, engine='openpyxl', mode='a') as writer:
        df2.to_excel(writer, sheet_name='Power', index=False)
        e_df.to_excel(writer, sheet_name='Bus Energy', index=True)

    # Create DataFrame for the thrid set of data
    dataPTO = {
        'Costs_PTO':[pyo.value(costs)],
        'sell_PTO': [pyo.value(sell)],
        'buy_PTO': [pyo.value(buy)],
        'degra_PTO': [pyo.value(degra)],
        'power_PTO': [pyo.value(power)],
        'cap_PTO': [pyo.value(cap)],
    }

    dataAggre = {
        'revenues_Agg': [pyo.value(revenues)],
        'sell_to_PTO_Agg': [pyo.value(sell_to_PTO)],
        'buy_from_PTO_Agg': [pyo.value(buy_from_PTO)],
        'cap_from_PTO_Agg': [pyo.value(cap_from_PTO)],
        'cap_to_GRID_Agg': [pyo.value(cap_to_GRID)],
        'buy_from_GRID_Agg': [pyo.value(buy_from_GRID)],
        'sell_to_GRID_Agg': [pyo.value(sell_to_GRID)]
    }

    dfPTO = pd.DataFrame(dataPTO)
    dfAggre= pd.DataFrame(dataAggre)

    # Save the second set of DataFrames to the same Excel file, each in a separate sheet
    with pd.ExcelWriter(output_file_name, engine='openpyxl', mode='a') as writer:
        dfPTO.to_excel(writer, sheet_name='PTO', index=False)
        dfAggre.to_excel(writer, sheet_name='Aggregator', index=False)

## SOLVING

In [6]:
for scenario in range(1,5):
    print('Scenario number:',scenario)
    UB = pyo.value('inf')
    LB = float('-inf')
    k = 1
    y_buy_l = 0
    y_sell_l = 0
    y_cap_l = 0
    d_l = 0
    u_l = 0
    epsilon = 0.0001
    while(UB != LB):
        model_HRP = solveHRP(data,y_buy_l,y_sell_l,y_cap_l, d_l,u_l,k,scenario)
        pho_plus = model_HRP.pho_plus
        if scenario == 2 or 4:
            pho_minus = model_HRP.pho_minus
            y_buy_u = model_HRP.w_buy
        if scenario == 3 or 4:
            mi = model_HRP.mi
            y_cap_u = model_HRP.w_cap
        y_sell_u = model_HRP.w_sell
        UB = model_HRP.obj()
        
        model_LL = solveLL(data,pho_plus,pho_minus,mi,scenario)
        y_buy_l = model_LL.w_buy
        if scenario == 2 or 4:
            y_sell_l = model_LL.w_sell
            d_l = model_LL.d
        if scenario == 3 or 4:
            y_cap_l = model_LL.w_cap
        u_l = model_LL.u
        
        f_y_u = sum(model_LL.pho_plus[p]*y_buy_u[t] for p in model_LL.P for t in range(model_LL.Q_begin[p], model_LL.Q_end[p])) - sum(model_LL.pho_minus[p]*y_sell_u[t] for p in model_LL.P for t in range(model_LL.Q_begin[p], model_LL.Q_end[p])) + sum(model_LL.d[k, t] for k in model_LL.K for t in model_LL.T) - sum(model_LL.mi[p]*y_cap_u[t] for p in model_LL.P for t in range(model_LL.Q_begin[p], model_LL.Q_end[p])) + sum(model_LL.U_price[l]*model_LL.u[l] for l in model_LL.L)
        if pyo.value(f_y_u) - model_LL.obj() <= epsilon:
            LB = UB
            visualization(model_LL, model_HRP,pho_plus,pho_minus,mi,y_buy_l,y_sell_l,y_cap_l,scenario)
        else:
            F_y_l = sum(pho_plus[p] * y_buy_l[t] for p in model_HRP.P for t in range(model_HRP.Q_begin[p], model_HRP.Q_end[p])) - sum(pho_minus[p] * y_sell_l[t] for p in model_HRP.P for t in range(model_HRP.Q_begin[p], model_HRP.Q_end[p])) + sum(model_HRP.PI_cap[t] * y_cap_l[t] for p in model_HRP.P for t in range(model_HRP.Q_begin[p], model_HRP.Q_end[p])) - sum(mi[p] * y_cap_l[t] for p in model_HRP.P for t in range(model_HRP.Q_begin[p], model_HRP.Q_end[p])) + sum(model_HRP.PI[t] * y_sell_l[t] for t in model_HRP.T) - sum(model_HRP.PI[t] * y_buy_l[t] for t in model_HRP.T)

            if pyo.value(F_y_l) > LB:
                LB = pyo.value(F_y_l)
        k = k+1
        if k == 5:
            visualization(model_LL, model_HRP,pho_plus,pho_minus,mi,y_buy_l,y_sell_l,y_cap_l,scenario)
            break    

Scenario number: 1

Values from the PTO side:
Total costs: 25.02500179733475 
Energy bought from the Aggregator: 21.60000179733475 
Energy sold to the Aggregator: 0.0 
Battery degradation costs: 0.0 
Power bought: 3.425 
FCR offered to the Aggregator: 0.0 

Values from the Aggregator side:
Total revenues: -10.43909820266525 
Energy sold to the PTO: 21.60000179733475 
Energy bought from the PTO: 0.0 
FCR bought from the PTO: 0.0 
FCR offered to the grid: 0.0 
Energy bought in the Wholesale market: 32.0391 
Energy sold to the grid: 0.0
Scenario number: 2

Values from the PTO side:
Total costs: 17.335611354957702 
Energy bought from the Aggregator: 19.799998211414675 
Energy sold to the Aggregator: 6.765697286220227 
Battery degradation costs: 0.8763104297632563 
Power bought: 3.425 
FCR offered to the Aggregator: 0.0 

Values from the Aggregator side:
Total revenues: -15.629949074805554 
Energy sold to the PTO: 19.799998211414675 
Energy bought from the PTO: 6.765697286220227 
FCR bought